In [ ]:
!pip install imageio

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import imageio
from tensorflow.keras import layers, models



plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

# Edge Detection Examples

In this section, we'll explore basic edge detection filters, including vertical, horizontal, and diagonal edges. These filters help highlight specific types of edges in images and are commonly used in convolutional neural networks (CNNs) for feature extraction.

In [ ]:
# Load a sample grayscale image
# Experiment: Try different images to see how edge detection works on various subjects
img = cv2.imread('cube.png', cv2.IMREAD_GRAYSCALE)

# Define edge detection kernels
# Experiment: Modify these kernels to see how they affect edge detection
# You can try different values or even 3x3 kernels

vertical_kernel = np.array([[-1, 1],
                            [-1, 1]])

horizontal_kernel = np.array([[1, 1],
                              [-1, -1]])

# Function to perform convolution
def convolve2d(image, kernel):
    # Experiment: You can modify this function to add padding options
    # or change how the convolution is performed
    output = np.zeros_like(image)
    padded_image = np.pad(image, ((1, 1), (1, 1)), mode='edge')
    
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            output[i, j] = np.sum(padded_image[i:i+2, j:j+2] * kernel)
    
    return output

# Apply edge detection
vertical_edges = convolve2d(img, vertical_kernel)
horizontal_edges = convolve2d(img, horizontal_kernel)

# Normalize the output for visualization
# Experiment: Try different normalization methods or skip normalization
# to see how it affects the visibility of edges
vertical_edges_norm = cv2.normalize(vertical_edges, None, 0, 255, cv2.NORM_MINMAX)
horizontal_edges_norm = cv2.normalize(horizontal_edges, None, 0, 255, cv2.NORM_MINMAX)

plt.figure(figsize=(15, 5))

# Experiment: You can add more subplots to show intermediate steps
# or different variations of edge detection
plt.subplot(131)
plt.imshow(img, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(132)
plt.imshow(vertical_edges_norm, cmap='gray')
plt.title('Vertical Edge Detection')
plt.axis('off')

# plt.subplot(133)
# plt.imshow(horizontal_edges_norm, cmap='gray')
# plt.title('Horizontal Edge Detection')
# plt.axis('off')

combined_edges = np.sqrt(vertical_edges**2 + horizontal_edges**2)
plt.subplot(144)
plt.imshow(combined_edges, cmap='gray')
plt.title('Combined Edges')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:

img = cv2.imread('cube.png', cv2.IMREAD_GRAYSCALE)
# Function to perform 2D convolution
def convolve2d(image, kernel):
    output = np.zeros_like(image, dtype=float)
    pad_height = kernel.shape[0] // 2
    pad_width = kernel.shape[1] // 2
    padded_image = np.pad(image, ((pad_height, pad_height), (pad_width, pad_width)), mode='edge')
    
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            output[i, j] = np.sum(padded_image[i:i+kernel.shape[0], j:j+kernel.shape[1]] * kernel)
    
    return output

# Define kernels
identity_kernel = np.array([[0, 0, 0],
                            [0, 1, 0],
                            [0, 0, 0]])

blur_kernel = np.array([[1, 1, 1],
                        [1, 1, 1],
                        [1, 1, 1]]) / 0.00009

sharpen_kernel = np.array([[0, 2, 0],
                           [2, -8, 2],
                           [0, 2, 0]])

edge_detect_kernel = np.array([[1, 1, 1],
                               [1, -80, 1],
                               [1, 1, 1]])

# Apply convolutions
identity_result = convolve2d(img, identity_kernel)
blur_result = convolve2d(img, blur_kernel)
sharpen_result = convolve2d(img, sharpen_kernel)
edge_detect_result = convolve2d(img, edge_detect_kernel)

# Function to normalize and convert to uint8
def normalize_to_uint8(img):
    return ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)

# Normalize results
identity_result = normalize_to_uint8(identity_result)
blur_result = normalize_to_uint8(blur_result)
sharpen_result = normalize_to_uint8(sharpen_result)
edge_detect_result = normalize_to_uint8(edge_detect_result)

# Save images
cv2.imwrite('original_gradient.png', img)
cv2.imwrite('identity_result.png', identity_result)
cv2.imwrite('blur_result.png', blur_result)
cv2.imwrite('sharpen_result.png', sharpen_result)
cv2.imwrite('edge_detect_result.png', edge_detect_result)

# Display results for verification
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Convolution Operations on Gradient Image', fontsize=16)

axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 1].imshow(identity_result, cmap='gray')
axes[0, 1].set_title('Identity')
axes[0, 2].imshow(blur_result, cmap='gray')
axes[0, 2].set_title('Blur')
axes[1, 0].imshow(sharpen_result, cmap='gray')
axes[1, 0].set_title('Sharpen')
axes[1, 1].imshow(edge_detect_result, cmap='gray')
axes[1, 1].set_title('Edge Detection')

for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Create a smaller sample image (8x8)
img = np.random.randint(0, 255, (8, 8), dtype=np.uint8)

# Define a simple kernel
kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

def visualize_convolution(image, kernel, sample_rate=2):
    output = np.zeros((image.shape[0] - kernel.shape[0] + 1, 
                       image.shape[1] - kernel.shape[1] + 1))
    
    steps = []
    step = 1
    
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            roi = image[i:i+kernel.shape[0], j:j+kernel.shape[1]]
            output[i, j] = np.sum(roi * kernel)
            
            if step % sample_rate == 0 or step == 1 or step == output.size:
                fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
                
                ax1.imshow(image, cmap='gray', vmin=0, vmax=255)
                ax1.set_title('Input Image')
                ax1.axis('off')
                
                ax2.imshow(image, cmap='gray', vmin=0, vmax=255)
                ax2.set_title(f'Kernel Position (Step {step})')
                rect = plt.Rectangle((j-0.5, i-0.5), kernel.shape[1], kernel.shape[0], 
                                     fill=False, edgecolor='red', linewidth=2)
                ax2.add_patch(rect)
                ax2.axis('off')
                
                ax3.imshow(output, cmap='gray', vmin=output.min(), vmax=output.max())
                ax3.set_title('Output')
                ax3.axis('off')
                
                plt.tight_layout()
                
                # Instead of saving, store the figure in memory
                fig.canvas.draw()
                image_from_plot = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
                image_from_plot = image_from_plot.reshape(fig.canvas.get_width_height()[::-1] + (3,))
                steps.append(image_from_plot)
                
                plt.close(fig)
            
            step += 1
    
    return steps

# Run the visualization
animation_frames = visualize_convolution(img, kernel)

# Create a GIF
output_filename = 'convolution_animation.gif'
imageio.mimsave(output_filename, animation_frames, duration=0.5)

print(f"Animation saved as {output_filename}")

# Convolutional Layers
## Implementing Convolutional Layers With Keras

Let's load two sample images, rescale their pixel values to 0-1, and center crop them to small 70×120 images:

In [ ]:
from sklearn.datasets import load_sample_images
import tensorflow as tf

images = load_sample_images()["images"]
images = tf.keras.layers.CenterCrop(height=70, width=120)(images)
images = tf.keras.layers.Rescaling(scale=1 / 255)(images)

In [ ]:
images.shape

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
conv_layer = tf.keras.layers.Conv2D(filters=2, kernel_size=1)
fmaps = conv_layer(images)

In [ ]:
fmaps.shape

In [ ]:
# extra code – displays the two output feature maps for each image

plt.figure(figsize=(15, 9))
for image_idx in (0, 1):
    for fmap_idx in (0, 1):
        plt.subplot(2, 2, image_idx * 2 + fmap_idx + 1)
        plt.imshow(fmaps[image_idx, :, :, fmap_idx], cmap="gray")
        plt.axis("off")

plt.show()

As you can see, randomly generated filters typically act like edge detectors, which is great since that's a useful tool in image processing, and that's the type of filters that a convolutional layer typically starts with. Then, during training, it gradually learns improved filters to recognize useful patterns for the task.

Now let's use zero-padding:

In [ ]:
conv_layer = tf.keras.layers.Conv2D(filters=32, kernel_size=7,
                                    padding="same")
fmaps = conv_layer(images)

In [ ]:
fmaps.shape

In [ ]:
# extra code – shows that the output shape when we set strides=2
conv_layer = tf.keras.layers.Conv2D(filters=32, kernel_size=7, padding="same",
                                    strides=2)
fmaps = conv_layer(images)
fmaps.shape

In [ ]:
# extra code – this utility function can be useful to compute the size of the
#              feature maps output by a convolutional layer. It also returns
#              the number of ignored rows or columns if padding="valid", or the
#              number of zero-padded rows or columns if padding="same"."""

import numpy as np

def conv_output_size(input_size, kernel_size, strides=1, padding="valid"):
    if padding=="valid":
        z = input_size - kernel_size + strides
        output_size = z // strides
        num_ignored = z % strides
        return output_size, num_ignored
    else:
        output_size = (input_size - 1) // strides + 1
        num_padded = (output_size - 1) * strides + kernel_size - input_size
        return output_size, num_padded

conv_output_size(np.array([70, 120]), kernel_size=7, strides=2, padding="same")

Let's now look at the weights:

In [ ]:
kernels, biases = conv_layer.get_weights()
kernels.shape

In [ ]:
biases.shape

In [ ]:
# extra code – shows how to use the tf.nn.conv2d() operation

tf.random.set_seed(42)
filters = tf.random.normal([7, 7, 3, 2])
biases = tf.zeros([2])
fmaps = tf.nn.conv2d(images, filters, strides=1, padding="SAME") + biases

Let's manually create two filters full of zeros, except for a vertical line of 1s in the first filter, and a horizontal one in the second filter (just like in Figure 14–5). The two output feature maps highlight vertical lines and horizontal lines, respectively. In practice you will probably never need to create filters manually, since the convolutional layers will learn them automatically.

In [ ]:
# extra code – shows how to manually create two filters to get images similar
#              to those in Figure 14–5.

plt.figure(figsize=(15, 9))
filters = np.zeros([7, 7, 3, 2])
filters[:, -3, :, 0] = 1
filters[-3, :, :, 1] = 1
fmaps = tf.nn.conv2d(images, filters, strides=1, padding="SAME") + biases

for image_idx in (0, 1):
    for fmap_idx in (0, 1):
        plt.subplot(2, 2, image_idx * 2 + fmap_idx + 1)
        plt.imshow(fmaps[image_idx, :, :, fmap_idx], cmap="gray")
        plt.axis("off")

plt.show()

Notice the dark lines at the top and bottom of the two images on the left, and on the left and right of the two images on the right? Can you guess what these are? Why were they not present in the previous figure?

You guessed it! These are artifacts due to the fact that we used zero padding in this case, while we did not use zero padding to create the feature maps in the previous figure. Because of zero padding, the two feature maps based on the vertical line filter (i.e., the two left images) could not fully activate near the top and bottom of the images. Similarly, the two feature maps based on the horizontal line filter (i.e., the two right images) could not fully activate near the left and right of the images.

# Pooling Layers
## Implementing Pooling Layers With Keras

**Max pooling**

In [ ]:
max_pool = tf.keras.layers.AvgPool2D(pool_size=4)

In [ ]:
output = max_pool(images)

In [ ]:
# extra code – this cells shows what max pooling with stride = 2 looks like

import matplotlib as mpl

fig = plt.figure(figsize=(12, 8))
gs = mpl.gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[2, 1])

ax1 = fig.add_subplot(gs[0, 0])
ax1.set_title("Input")
ax1.imshow(images[0])  # plot the 1st image
ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title("Output")
ax2.imshow(output[0])  # plot the output for the 1st image
ax2.axis("off")
plt.show()

**Depth-wise pooling**

In [ ]:
# extra code – shows how to use the max_pool() op; only works on the CPU
np.random.seed(42)
fmaps = np.random.rand(2, 70, 120, 60)
with tf.device("/cpu:0"):
    output = tf.nn.max_pool(fmaps, ksize=(1, 1, 1, 3), strides=(1, 1, 1, 3),
                            padding="VALID")
output.shape

In [ ]:
class DepthPool(tf.keras.layers.Layer):
    def __init__(self, pool_size=2, **kwargs):
        super().__init__(**kwargs)
        self.pool_size = pool_size
    
    def call(self, inputs):
        shape = tf.shape(inputs)  # shape[-1] is the number of channels
        groups = shape[-1] // self.pool_size  # number of channel groups
        new_shape = tf.concat([shape[:-1], [groups, self.pool_size]], axis=0)
        return tf.reduce_max(tf.reshape(inputs, new_shape), axis=-1)

In [ ]:
# extra code – shows that this custom layer gives the same result as max_pool()
np.allclose(DepthPool(pool_size=3)(fmaps), output)

In [ ]:
# extra code – computes and displays the output of the depthwise pooling layer

depth_output = DepthPool(pool_size=3)(images)

plt.figure(figsize=(12, 8))
plt.subplot(1, 2, 1)
plt.title("Input")
plt.imshow(images[0])  # plot the 1st image
plt.axis("off")
plt.subplot(1, 2, 2)
plt.title("Output")
plt.imshow(depth_output[0, ..., 0], cmap="gray")  # plot 1st image's output
plt.axis("off")
plt.show()

**Global Average Pooling**

In [ ]:
global_avg_pool = tf.keras.layers.GlobalAvgPool2D()

The following layer is equivalent:

In [ ]:
global_avg_pool = tf.keras.layers.Lambda(
    lambda X: tf.reduce_mean(X, axis=[1, 2]))

In [ ]:
global_avg_pool(images)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio



# Generate a small sample image (8x8)
# Warning: Using a real image is not advised, your computer will probably run out of memory and vscode will crash
img = np.random.randint(0, 255, (8, 8), dtype=np.uint8)

# Define a simple kernel
kernel = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]])

def visualize_padding(image, kernel, padding_size=8):
    output_no_padding = np.zeros((image.shape[0] - kernel.shape[0] + 1, 
                                  image.shape[1] - kernel.shape[1] + 1))
    output_with_padding = np.zeros((image.shape[0] + 2 * padding_size - kernel.shape[0] + 1, 
                                    image.shape[1] + 2 * padding_size - kernel.shape[1] + 1))

    # Pad the image
    padded_image = np.pad(image, pad_width=padding_size, mode='edge')
    
    # Store frames for the animation
    steps = []

    # Convolution with and without padding
    for i in range(max(output_no_padding.shape[0], output_with_padding.shape[0])):
        for j in range(max(output_no_padding.shape[1], output_with_padding.shape[1])):
            # Convolution without padding
            if i < output_no_padding.shape[0] and j < output_no_padding.shape[1]:
                roi = image[i:i+kernel.shape[0], j:j+kernel.shape[1]]
                output_no_padding[i, j] = np.sum(roi * kernel)

            # Convolution with padding
            if i < output_with_padding.shape[0] and j < output_with_padding.shape[1]:
                roi = padded_image[i:i+kernel.shape[0], j:j+kernel.shape[1]]
                output_with_padding[i, j] = np.sum(roi * kernel)

            # Visualize this step
            fig, axes = plt.subplots(2, 2, figsize=(10, 10))

            # Show input image
            axes[0, 0].imshow(image, cmap='gray', vmin=0, vmax=255)
            axes[0, 0].set_title('Input Image')
            axes[0, 0].axis('off')

            # Show padded image
            axes[0, 1].imshow(padded_image, cmap='gray', vmin=0, vmax=255)
            axes[0, 1].set_title('Padded Image')
            axes[0, 1].axis('off')

            # Show output without padding
            axes[1, 0].imshow(output_no_padding, cmap='gray', vmin=output_no_padding.min(), vmax=output_no_padding.max())
            axes[1, 0].set_title('Output (No Padding)')
            axes[1, 0].axis('off')

            # Show output with padding
            axes[1, 1].imshow(output_with_padding, cmap='gray', vmin=output_with_padding.min(), vmax=output_with_padding.max())
            axes[1, 1].set_title('Output (With Padding)')
            axes[1, 1].axis('off')

            plt.tight_layout()
            
            # Save the figure to array
            fig.canvas.draw()
            image_from_plot = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
            image_from_plot = image_from_plot.reshape(fig.canvas.get_width_height()[::-1] + (3,))
            steps.append(image_from_plot)

            plt.close(fig)

    return steps

# Run the visualization for padding
animation_frames = visualize_padding(img, kernel)

# Create a GIF
output_filename = 'padding_animation.gif'
imageio.mimsave(output_filename, animation_frames, duration=0.5)

print(f"Animation saved as {output_filename}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio

# Generate a small sample image (8x8)
# Warning: Using a real image is not advised, your computer will probably run out of memory and vscode will crash
img = np.random.randint(0, 255, (64, 64), dtype=np.uint8)

# Define pooling size
pool_size = (4, 4)

def visualize_pooling(image, pool_size):
    output = np.zeros((image.shape[0] // pool_size[0], image.shape[1] // pool_size[1]))
    
    # Store frames for the animation
    steps = []

    # Max pooling
    for i in range(0, image.shape[0], pool_size[0]):
        for j in range(0, image.shape[1], pool_size[1]):
            
            
            pool = image[i:i+pool_size[0], j:j+pool_size[1]]
            output[i//pool_size[0], j//pool_size[1]] = np.max(pool)

            # Visualize this step
            fig, axes = plt.subplots(1, 2, figsize=(10, 5))

            # Show input image
            axes[0].imshow(image, cmap='gray', vmin=0, vmax=255)
            axes[0].set_title('Input Image')
            axes[0].axis('off')

            # Highlight the current pooling region
            rect = plt.Rectangle((j - 0.5, i - 0.5), pool_size[1], pool_size[0], fill=False, edgecolor='red', linewidth=2)
            axes[0].add_patch(rect)

            # Show output
            axes[1].imshow(output, cmap='gray', vmin=0, vmax=255)
            axes[1].set_title('Max Pooling Output')
            axes[1].axis('off')

            plt.tight_layout()
            
            # Save the figure to array
            fig.canvas.draw()
            image_from_plot = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
            image_from_plot = image_from_plot.reshape(fig.canvas.get_width_height()[::-1] + (3,))
            steps.append(image_from_plot)

            plt.close(fig)

    return steps

# Run the visualization for pooling
animation_frames = visualize_pooling(img, pool_size)

# Create a GIF
output_filename = 'pooling_animation.gif'
imageio.mimsave(output_filename, animation_frames, duration=0.5)

print(f"Animation saved as {output_filename}")